### Ejercicio 1 – Precio promedio por barrio

Enunciado:

Agrupa por barrio y calcula el precio medio de las viviendas. ¿Qué barrio es el más caro en promedio?

In [7]:
import pandas as pd
import numpy as np

In [8]:
df=pd.read_csv('Dataset_vivienda_en_Madrid.csv')
df.head(5)

,barrio,metros_cuadrados,habitaciones,baños,precio,latitud,longitud
0,Salamanca,138,2,3,580263,40.431979,-3.680183
1,Arganzuela,113,1,3,461554,40.400009,-3.695814
2,Chamartín,118,5,1,573500,40.463564,-3.677655
3,Villa de Vallecas,106,5,1,514179,40.366842,-3.603498
4,Tetuán,139,5,1,571224,40.457287,-3.704847


In [9]:
df_agrupado=df.groupby("barrio")["precio"].mean().sort_values(ascending=False)
df_agrupado

barrio
Arganzuela            532983.166667
Carabanchel           525437.800000
Salamanca             510747.416667
Latina                508566.681818
Puente de Vallecas    507813.818182
Moratalaz             498366.000000
Usera                 496345.913043
Retiro                485115.428571
Villa de Vallecas     483566.736842
Tetuán                478552.652174
Chamartín             465584.941176
Centro                464953.312500
Hortaleza             449413.444444
Moncloa               408381.045455
San Blas              394232.538462
Name: precio, dtype: float64

### Ejercicio 2 – Precio medio por metro cuadrado

Enunciado:

Agrega una nueva columna precio_m2 y calcula el promedio de este valor por barrio. Usa groupby + apply.

In [12]:
df["precio_m2"] = df["precio"] / df["metros_cuadrados"]
df.head(5)


,barrio,metros_cuadrados,habitaciones,baños,precio,latitud,longitud,precio_m2
0,Salamanca,138,2,3,580263,40.431979,-3.680183,4204.804348
1,Arganzuela,113,1,3,461554,40.400009,-3.695814,4084.548673
2,Chamartín,118,5,1,573500,40.463564,-3.677655,4860.169492
3,Villa de Vallecas,106,5,1,514179,40.366842,-3.603498,4850.745283
4,Tetuán,139,5,1,571224,40.457287,-3.704847,4109.525180


In [13]:
df.groupby("barrio")["precio_m2"].apply(np.mean).sort_values(ascending=False)

barrio
San Blas              4824.452236
Moncloa               4473.865672
Retiro                4447.187923
Tetuán                4437.854501
Moratalaz             4372.732787
Centro                4366.268235
Usera                 4343.341824
Latina                4332.705016
Salamanca             4277.329952
Villa de Vallecas     4259.856867
Carabanchel           4254.644882
Hortaleza             4254.300270
Puente de Vallecas    4222.200587
Arganzuela            4208.727815
Chamartín             4140.651250
Name: precio_m2, dtype: float64

### Ejercicio 3 – Clasificación por cuartiles de precio

Enunciado:

Agrupa por barrio, calcula el precio promedio y asigna a cada barrio un cuartil (Q1–Q4) según el valor medio.

In [ ]:
precio_prom = df.groupby("barrio")["precio"].mean()
cuartiles = pd.qcut(precio_prom, q=4, labels=["Q1", "Q2", "Q3", "Q4"])
df["cuartil_barrio"] = df["barrio"].map(cuartiles)
df.head(10)

,barrio,metros_cuadrados,habitaciones,baños,precio,latitud,longitud,precio_m2,cuartil_barrio
0,Salamanca,138,2,3,580263,40.431979,-3.680183,4204.804348,Q4
1,Arganzuela,113,1,3,461554,40.400009,-3.695814,4084.548673,Q4
2,Chamartín,118,5,1,573500,40.463564,-3.677655,4860.169492,Q2
3,Villa de Vallecas,106,5,1,514179,40.366842,-3.603498,4850.745283,Q2
4,Tetuán,139,5,1,571224,40.457287,-3.704847,4109.525180,Q2
...,...,...,...,...,...,...,...,...,...
95,Villa de Vallecas,93,2,3,387573,40.367968,-3.603770,4167.451613,Q2
96,Usera,140,1,1,577913,40.382705,-3.709345,4127.950000,Q3
97,Centro,179,3,1,701589,40.415013,-3.701224,3919.491620,Q1
98,Puente de Vallecas,71,2,2,323315,40.392462,-3.659745,4553.732394,Q3


### Ejercicio 4 – Detectar viviendas “anómalas” en su barrio

Enunciado:

Una vivienda es “anómala” si su precio se aleja más de 2 desviaciones estándar del promedio de su barrio. Crea una nueva columna que indique esto.

In [17]:
def detectar_anomalias(grupo):
    media = grupo["precio"].mean()
    std = grupo["precio"].std()
    return grupo["precio"].apply(lambda x: abs(x - media) > 2 * std)



In [23]:
df["anomalia"] = df.groupby("barrio").apply(detectar_anomalias).reset_index(drop=True)
df_anomalas = df[df["anomalia"] == True]
df_anomalas

,barrio,metros_cuadrados,habitaciones,baños,precio,latitud,longitud,precio_m2,cuartil_barrio,anomalia
69,Hortaleza,148,2,1,567510,40.477466,-3.641553,3834.527027,Q1,True
81,Usera,119,4,3,554275,40.381551,-3.707143,4657.773109,Q3,True
110,Moratalaz,67,2,2,310383,40.404577,-3.653694,4632.582090,Q3,True


### Ejercicio 5 – Mapa con folium: precios por color

Enunciado:

Crea un mapa con folium donde cada vivienda tenga un color según su precio:

Verde: <300.000 €

Naranja: 300.000–600.000 €

Rojo: >600.000 €

In [ ]:
import folium

def color_precio(p):
    if p < 300000:
        return "green"
    elif p < 600000:
        return "orange"
    else:
        return "red"

mapa = folium.Map(location=[40.4168, -3.7038], zoom_start=12)



In [ ]:

for _, fila in df.iterrows():
    folium.CircleMarker(
        location=[fila['latitud'], fila['longitud']],
        radius=5,
        color=color_precio(fila['precio']),
        fill=True,
        fill_opacity=0.6,
        popup=f"{fila['barrio']} - {fila['precio']} €"
    ).add_to(mapa)

mapa.save("mapa_colores_precio.html")

### Ejercicio 6 – Mapa de calor de viviendas caras

Enunciado:

Crea un mapa de calor con folium.plugins.HeatMap que indique la concentración de viviendas con precio superior a 600.000 €.

In [29]:
from folium.plugins import HeatMap

mapa_calor = folium.Map(location=[40.4168, -3.7038], zoom_start=12)#localiacion madrid
data_heat = df[df["precio"] > 600000][["latitud", "longitud"]].values.tolist()
data_heat



[[40.40509371037048, -3.6557962871090783],
 [40.39346533776347, -3.745065258255051],
 [40.43061025608132, -3.62352137559689],
 [40.4168787689252, -3.702800252699016],
 [40.41550571206243, -3.6844877490284578],
 [40.42958116465288, -3.6763344945590872],
 [40.46319385200736, -3.679090575020849],
 [40.45772433248764, -3.702267228033784],
 [40.46329305275488, -3.678363425635328],
 [40.40991038480684, -3.657152221946781],
 [40.4166760050241, -3.704918745052863],
 [40.40568407148945, -3.657067324803368],
 [40.40830985536648, -3.651976618400501],
 [40.40136833397701, -3.6962975273361134],
 [40.43278427928036, -3.624303881726642],
 [40.43033998521466, -3.678412858985231],
 [40.40995252650508, -3.655715250092893],
 [40.385230594761374, -3.7049831317825257],
 [40.39593584598645, -3.7484571924696057],
 [40.381397111164354, -3.708735456239956],
 [40.38200639394892, -3.7471069152421],
 [40.393408718695895, -3.747863788477262],
 [40.41501304631147, -3.7012238250536895],
 [40.39981594489424, -3.69443

In [ ]:
HeatMap(data_heat, radius=15, blur=10).add_to(mapa_calor)
mapa_calor.save("mapa_calor_viviendas_caras.html")

### Ejercicio 7 – Vivienda más barata por barrio

Enunciado:

Encuentra la vivienda más barata de cada barrio (precio mínimo). Muestra su ubicación en un mapa.

In [27]:
viviendas_baratas = df.loc[df.groupby("barrio")["precio"].idxmin()]
viviendas_baratas




,barrio,metros_cuadrados,habitaciones,baños,precio,latitud,longitud,precio_m2,cuartil_barrio,anomalia
49,Arganzuela,68,1,1,280321,40.399485,-3.696941,4122.367647,Q4,False
113,Carabanchel,65,1,1,257049,40.385351,-3.746175,3954.600000,Q4,False
221,Centro,50,1,3,249504,40.415773,-3.703681,4990.080000,Q1,False
42,Chamartín,43,1,1,193223,40.464675,-3.676431,4493.558140,Q2,False
84,Hortaleza,54,1,2,191782,40.472032,-3.639939,3551.518519,Q1,False
143,Latina,42,1,2,183732,40.393478,-3.743338,4374.571429,Q4,False
60,Moncloa,46,3,1,170900,40.431853,-3.717174,3715.217391,Q1,False
27,Moratalaz,50,5,1,247324,40.407457,-3.655199,4946.480000,Q3,False
98,Puente de Vallecas,71,2,2,323315,40.392462,-3.659745,4553.732394,Q3,False
209,Retiro,54,2,1,200475,40.415968,-3.685210,3712.500000,Q2,False


In [28]:
mapa_min = folium.Map(location=[40.4168, -3.7038], zoom_start=12)
for _, row in viviendas_baratas.iterrows():
    folium.Marker(
        location=[row["latitud"], row["longitud"]],
        popup=f"{row['barrio']} - {row['precio']} €",
        icon=folium.Icon(color="blue", icon="home")
    ).add_to(mapa_min)

mapa_min.save("mapa_viviendas_baratas.html")